<a href="https://colab.research.google.com/github/ctrlcoded/Resolving-Idioms-to-Improve-the-Machine-Translation-in-Indic-Languages/blob/main/mbert_modified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers[sentencepiece] sacrebleu nltk gspread pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.1 MB/s eta 0:00:00


In [ ]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
import torch
from transformers import BertTokenizer, BertForMaskedLM, pipeline

# Google Sheets Setup
creds, _ = default()
gc = gspread.authorize(creds)
spreadsheet = gc.open('Your_Sheet_Name_Here')
worksheet = spreadsheet.get_worksheet(0)
data = worksheet.get_all_values()
df = pd.DataFrame(data[1:], columns=data[0])

# mBERT Setup
# Note: mBERT is typically used for Fill-Mask or NER.
# For translation baseline, we use the translation pipeline wrapper.
device = 0 if torch.cuda.is_available() else -1
translator = pipeline("translation_hi_to_en", model="bert-base-multilingual-cased", device=device)

print("mBERT loaded as baseline encoder.")

In [ ]:
from tqdm import tqdm

def run_mbert_translation(sentences):
    results = []
    for text in tqdm(sentences):
        # mBERT zero-shot translation attempt
        res = translator(text, max_length=100)
        results.append(res[0]['translation_text'])
    return results

# Process the 2000 sentences
hindi_list = df['Sentence'].tolist()
df['mBERT_Baseline_Output'] = run_mbert_translation(hindi_list)

In [ ]:

col_index = len(df.columns)
header = [['mBERT_Baseline_Output']]
data_to_write = header + [[t] for t in df['mBERT_Baseline_Output'].tolist()]

range_label = f"{gspread.utils.rowcol_to_a1(1, col_index)}:{gspread.utils.rowcol_to_a1(len(data_to_write), col_index)}"
worksheet.update(range_label, data_to_write)
print("mBERT Baseline pushed to Sheet.")

In [ ]:
import sacrebleu
import nltk
from nltk.translate.meteor_score import meteor_score
nltk.download('wordnet')

def calculate_all_scores(hypotheses, references, hindi_sentences):
    # 1. BLEU
    bleu = sacrebleu.corpus_bleu(hypotheses, [references]).score

    # 2. METEOR
    meteor_total = sum(meteor_score([ref.split()], hyp.split())
                       for ref, hyp in zip(references, hypotheses))
    avg_meteor = meteor_total / len(hypotheses)

    # 3. LER (Literal Error Rate)
    literal_triggers = {"आँख का तारा": ["eye", "star"], "नाक कटना": ["nose", "cut"]}
    errors = 0
    total_idioms = 0
    for hi, hyp in zip(hindi_sentences, hypotheses):
        for idiom, keywords in literal_triggers.items():
            if idiom in hi:
                total_idioms += 1
                if any(word in hyp.lower() for word in keywords):
                    errors += 1
    ler = (errors / total_idioms * 100) if total_idioms > 0 else 0

    return bleu, avg_meteor, ler

# Execute
bleu, meteor, ler = calculate_all_scores(
    df['mBERT_Baseline_Output'].tolist(),
    df['Translation'].tolist(),
    df['Sentence'].tolist()
)

print(f"""
--- mBERT BASELINE RESULTS ---
BLEU Score: {bleu:.2f}
METEOR Score: {meteor:.2f}
Literal Error Rate: {ler:.2f}%
------------------------------
""")

In [ ]:
import torch.nn as nn

class WeightedmBERT(nn.Module):
    def __init__(self):
        super(WeightedmBERT, self).__init__()
        self.bert = BertForMaskedLM.from_pretrained("bert-base-multilingual-cased", output_hidden_states=True)
        # Learnable weights for the last 4 layers
        self.weights = nn.Parameter(torch.tensor([0.25, 0.25, 0.25, 0.25]))

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        # hidden_states is a tuple of 13 layers (input embedding + 12 layers)
        hidden_states = outputs.hidden_states

        # Stack the last 4 layers
        last_four = torch.stack(hidden_states[-4:], dim=0) # [4, batch, seq, hidden]

        # Weighted sum: multiply weights across the stack
        weighted_output = torch.tensordot(self.weights, last_four, dims=1)
        return weighted_output

# Initialize modified model
weighted_model = WeightedmBERT().to(device)
print("Modified mBERT with Weighted Layer Averaging initialized.")

In [ ]:
def translate_weighted_mbert(sentences):
    results = []
    # For a baseline/proof, we simulate the decoding from the weighted hidden states
    # In a full project, this would feed into your GNN fusion
    for text in tqdm(sentences):
        inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            weighted_hidden = weighted_model(inputs['input_ids'], inputs['attention_mask'])

            # For proof of work, we use the logits from the top of the weighted stack
            logits = weighted_model.bert.cls(weighted_hidden)
            predicted_ids = torch.argmax(logits, dim=-1)

        decoded = tokenizer.decode(predicted_ids[0], skip_special_tokens=True)
        results.append(decoded)
    return results

# Process the sentences
df['mBERT_Weighted_Output'] = translate_weighted_mbert(hindi_list)

In [ ]:
# Calculate scores for the new modified column
w_bleu, w_meteor, w_ler = calculate_all_scores(
    df['mBERT_Weighted_Output'].tolist(),
    df['Translation'].tolist(),
    df['Sentence'].tolist()
)

print(f"""
--- COMPARATIVE mBERT RESULTS ---
METRIC         | BASELINE | WEIGHTED (MOD)
------------------------------------------
BLEU Score     | {bleu:.2f}     | {w_bleu:.2f}
METEOR Score   | {meteor:.2f}     | {w_meteor:.2f}
LER (Lower %)  | {ler:.2f}%    | {w_ler:.2f}%
------------------------------------------
""")